# Body Fat: annealed VIMCO-GM training

Collapsed spike-and-slab regression with $\tau^2=1$. Run from this folder. Fits every tuning configuration for every optimisation seed (12345-12394) and writes PIPs, logits and run metadata to `the output folder`.

In [ ]:
%load_ext Cython


In [ ]:
%%cython --compile-args=-O3
# cython: language_level=3
# cython: boundscheck=False
# cython: wraparound=False
# cython: cdivision=True
# cython: initializedcheck=False

import os
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

device = torch.device('cpu')
dtype = torch.float64

METHOD_SEEDS = range(12345, 12395)
METHOD_NAME = 'VIMCO-GM annealed'

tau2 = 1.0
prior_prec = 1.0 / tau2
jitter = 1e-8
import os
BODYFAT_M0 = float(os.environ.get('BODYFAT_M0', '3'))   # 3 = main analysis; 1.5 or 6 = Appendix F
M0_TAG = f'{BODYFAT_M0:g}'.replace('.', 'p')
PRIOR_EXPECTED_MODEL_SIZE = BODYFAT_M0
b_pi = 1.0
a_pi = None

TOTAL_SAMPLES = 300
S_COPIES_THIS_FILE = 1
N_IMPORTANCE_THIS_FILE = 300
split_label = f'S_{S_COPIES_THIS_FILE}_N_{N_IMPORTANCE_THIS_FILE}'

LR_GRID = [0.01, 0.05, 0.1, 0.2, 0.3]
ALPHA_START = 0.99
ALPHA_MIN = 0.0
ALPHA_STEP = 0.01
ESS_LOW = 0.8 * TOTAL_SAMPLES

PATIENCE_MAX = 200
MAX_ITER = 3000
BOUND_TOL = 1e-4
MAX_GRAD_NORM = 5.0

ADAPTIVE_ACTIVE_MAX_Q = 50
ACTIVE_CHOL_BATCH_SIZE = 64
DUAL_FALLBACK_BATCH_SIZE = 8

ROOT = Path.cwd().resolve()
output_base_root = ROOT / f'output_bodyfat_m0_{M0_TAG}_methodseeds_50_no_truth'
output_root = output_base_root / f'm0_{M0_TAG}'
output_root.mkdir(parents=True, exist_ok=True)

NUM_TORCH_THREADS = 2
torch.set_num_threads(NUM_TORCH_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

METHOD_SEEDS_LIST = list(METHOD_SEEDS)
if not METHOD_SEEDS_LIST or len(set(METHOD_SEEDS_LIST)) != len(METHOD_SEEDS_LIST):
    raise ValueError('METHOD_SEEDS must contain distinct optimisation seeds.')

DATA_FILE_CANDIDATES = [
    ROOT / 'bodyfat.csv',
    ROOT / 'bodyfat' / 'bodyfat.csv',
    ROOT.parent / 'bodyfat' / 'bodyfat.csv',
]
bodyfat_data_file = next((path for path in DATA_FILE_CANDIDATES if path.is_file()), None)
if bodyfat_data_file is None:
    raise FileNotFoundError('Cannot find bodyfat.csv. Tried:\n' +
                            '\n'.join(str(path) for path in DATA_FILE_CANDIDATES))
_header_df = pd.read_csv(bodyfat_data_file, nrows=1)
response_col = next((c for c in ['y', 'body_fat', 'brozek_fat', 'bodyfat',
                                'percent_body_fat'] if c in _header_df), None)
if response_col is None:
    raise ValueError('bodyfat.csv must contain a response column named y or body_fat.')
feature_cols = [c for c in _header_df if c != response_col and c != 'case_id'
                and not str(c).startswith('Unnamed')]
p = len(feature_cols)
if p != 13:
    raise ValueError(f'Expected 13 Body Fat predictors, found {p}.')
a_pi = b_pi * PRIOR_EXPECTED_MODEL_SIZE / (p - PRIOR_EXPECTED_MODEL_SIZE)
print('data_file =', bodyfat_data_file)
print('method =', METHOD_NAME, '|', split_label)
print('method seeds =', METHOD_SEEDS_LIST)
print('X is standardised internally using ddof=0; y remains on its original scale.')
print('a_pi =', a_pi, '| b_pi =', b_pi, '| tau2 =', tau2)
print('output_root =', output_root)

log_2pi_t = float(np.log(2.0 * np.pi))
tau2_t = torch.tensor(tau2, dtype=dtype, device=device)
prior_prec_t = torch.tensor(prior_prec, dtype=dtype, device=device)
log_prior_prec_t = torch.log(prior_prec_t)

a_pi_t = torch.tensor(a_pi, dtype=dtype, device=device)
b_pi_t = torch.tensor(b_pi, dtype=dtype, device=device)
p_ab_t = torch.tensor(float(p + a_pi + b_pi), dtype=dtype, device=device)
LOG_BETA_AB_t = torch.lgamma(a_pi_t) + torch.lgamma(b_pi_t) - torch.lgamma(a_pi_t + b_pi_t)
_MODEL_SIZE_GRID_t = torch.arange(p + 1, dtype=dtype, device=device)
_LOG_MODEL_PRIOR_BY_SIZE_t = (
    torch.lgamma(a_pi_t + _MODEL_SIZE_GRID_t)
    + torch.lgamma(b_pi_t + p - _MODEL_SIZE_GRID_t)
    - torch.lgamma(p_ab_t)
    - LOG_BETA_AB_t
)

n = None
nu_t = None
marginal_constant_t = None
XtX_t = None
Xty_t = None
yty_t = None
Xc_T_t = None
yc_t = None
ycyc_t = None
log_n_t = None
I_n_t = None
_REGULARISATION_CACHE = {}


def regularisation_matrix_cached(q):
    matrix = _REGULARISATION_CACHE.get(q)
    if matrix is None:
        prior_diag = torch.full((q,), prior_prec, dtype=dtype, device=device)
        prior_diag[0] = 0.0
        matrix = torch.diag(prior_diag) + jitter * torch.eye(q, dtype=dtype, device=device)
        _REGULARISATION_CACHE[q] = matrix
    return matrix

def load_dataset():
    global n, nu_t, marginal_constant_t
    global XtX_t, Xty_t, yty_t
    global Xc_T_t, yc_t, ycyc_t, log_n_t, I_n_t
    data_df = pd.read_csv(bodyfat_data_file)
    X_np = np.ascontiguousarray(data_df[feature_cols].to_numpy(np.float64))
    y_np = np.ascontiguousarray(data_df[response_col].to_numpy(np.float64))
    n = int(X_np.shape[0])
    if n < 3 or X_np.shape != (n, p):
        raise ValueError(f'Expected X=(n,{p}), y=(n,), with n>=3.')
    if not np.isfinite(X_np).all() or not np.isfinite(y_np).all():
        raise ValueError('Body Fat data contain non-finite values.')
    x_means = X_np.mean(axis=0)
    x_sds = X_np.std(axis=0, ddof=0)
    x_sds[x_sds == 0.0] = 1.0
    X_np -= x_means
    X_np /= x_sds
    nu_t = torch.tensor(float(n - 1), dtype=dtype, device=device)
    marginal_constant_t = torch.lgamma(0.5 * nu_t) - 0.5 * nu_t * log_2pi_t
    X_full_np = np.column_stack([np.ones(n, dtype=np.float64), X_np])
    XtX_t = torch.from_numpy(np.asarray(X_full_np.T @ X_full_np, np.float64)).to(device=device, dtype=dtype)
    Xty_t = torch.from_numpy(np.asarray(X_full_np.T @ y_np, np.float64)).to(device=device, dtype=dtype)
    yty_t = torch.tensor(float(y_np @ y_np), dtype=dtype, device=device)
    Xc_np = np.ascontiguousarray(X_np - X_np.mean(axis=0, keepdims=True))
    yc_np = np.ascontiguousarray(y_np - y_np.mean())
    Xc_T_t = torch.from_numpy(Xc_np).to(device=device, dtype=dtype).t().contiguous()
    yc_t = torch.from_numpy(yc_np).to(device=device, dtype=dtype)
    ycyc_t = torch.sum(yc_t.square())
    log_n_t = torch.log(torch.tensor(float(n), dtype=dtype, device=device))
    I_n_t = torch.eye(n, dtype=dtype, device=device)
    _REGULARISATION_CACHE.clear()
    return bodyfat_data_file

def gamma_from_eta(eta):
    return torch.sigmoid(eta)

def sample_G(gamma, total_samples):
    return torch.bernoulli(gamma.expand(total_samples, -1))

def log_q_eta(G, eta):
    return torch.sum(G * F.logsigmoid(eta) + (1.0 - G) * F.logsigmoid(-eta), dim=1)

def make_initial_eta():

    prior_mean = a_pi / (a_pi + b_pi)
    initial_lambda = np.log(prior_mean) - np.log1p(-prior_mean)
    return torch.full((p,), float(initial_lambda), dtype=dtype, device=device)

def log_model_prior_torch(G):
    model_size = G.sum(dim=1).to(torch.long)
    return _LOG_MODEL_PRIOR_BY_SIZE_t[model_size]

def log_marginal_likelihood_active_torch(G):
    number_models = int(G.shape[0])
    out = torch.empty(number_models, dtype=dtype, device=device)
    active_counts = G.sum(dim=1).to(torch.long)
    unique_counts = torch.unique(active_counts)

    for count_tensor in unique_counts:
        k_slope = int(count_tensor.item())
        row_indices_all = torch.nonzero(active_counts == k_slope, as_tuple=False).flatten()
        q = k_slope + 1

        for start in range(0, int(row_indices_all.numel()), ACTIVE_CHOL_BATCH_SIZE):
            row_indices = row_indices_all[start:start + ACTIVE_CHOL_BATCH_SIZE]
            batch_size = int(row_indices.numel())

            if k_slope == 0:
                idx_mat = torch.zeros((batch_size, 1), dtype=torch.long, device=device)
            else:
                group_G = G.index_select(0, row_indices)
                nonzero_locations = torch.nonzero(group_G > 0.5, as_tuple=False)
                slope_idx = nonzero_locations[:, 1].reshape(batch_size, k_slope).to(torch.long) + 1
                intercept = torch.zeros((batch_size, 1), dtype=torch.long, device=device)
                idx_mat = torch.cat((intercept, slope_idx), dim=1)

            XtX_batch = XtX_t[idx_mat[:, :, None], idx_mat[:, None, :]]
            Xty_batch = Xty_t[idx_mat]
            M_batch = XtX_batch + regularisation_matrix_cached(q).unsqueeze(0)
            L_batch = torch.linalg.cholesky(M_batch)
            solve_batch = torch.cholesky_solve(Xty_batch[..., None], L_batch).squeeze(-1)
            residual_batch = yty_t - torch.sum(Xty_batch * solve_batch, dim=1)
            residual_batch = torch.clamp(residual_batch, min=jitter)
            logdet_M_batch = 2.0 * torch.sum(
                torch.log(torch.diagonal(L_batch, dim1=-2, dim2=-1)), dim=1
            )
            out[row_indices] = (
                marginal_constant_t
                + 0.5 * k_slope * log_prior_prec_t
                - 0.5 * logdet_M_batch
                - 0.5 * nu_t * torch.log(residual_batch / 2.0)
            )

    return out

def log_marginal_likelihood_dual_internal_torch(G):
    number_models = int(G.shape[0])
    out = torch.empty(number_models, dtype=dtype, device=device)
    active_counts = G.sum(dim=1).to(torch.long)
    unique_counts = torch.unique(active_counts)

    for count_tensor in unique_counts:
        k_slope = int(count_tensor.item())
        row_indices = torch.nonzero(active_counts == k_slope, as_tuple=False).flatten()
        group_size = int(row_indices.numel())

        if k_slope == 0:
            residual = torch.clamp(ycyc_t, min=jitter)
            value = marginal_constant_t - 0.5 * log_n_t - 0.5 * nu_t * torch.log(residual / 2.0)
            out[row_indices] = value.expand(group_size)
            continue

        group_G = G.index_select(0, row_indices)
        nonzero_locations = torch.nonzero(group_G > 0.5, as_tuple=False)
        slope_idx = nonzero_locations[:, 1].reshape(group_size, k_slope).to(torch.long)

        for start in range(0, group_size, DUAL_FALLBACK_BATCH_SIZE):
            end = min(start + DUAL_FALLBACK_BATCH_SIZE, group_size)
            current_rows = row_indices[start:end]
            current_idx = slope_idx[start:end]
            current_batch_size = int(current_idx.shape[0])

            XA = Xc_T_t[current_idx].transpose(1, 2).contiguous()
            K_batch = I_n_t.unsqueeze(0) + tau2_t * torch.bmm(XA, XA.transpose(1, 2))
            if jitter > 0:
                K_batch = K_batch + jitter * I_n_t.unsqueeze(0)

            L_batch = torch.linalg.cholesky(K_batch)
            y_batch = yc_t.expand(current_batch_size, -1).unsqueeze(2)
            solved = torch.cholesky_solve(y_batch, L_batch).squeeze(2)
            residual_batch = torch.sum(yc_t.unsqueeze(0) * solved, dim=1)
            residual_batch = torch.clamp(residual_batch, min=jitter)
            logdet_K_batch = 2.0 * torch.sum(
                torch.log(torch.diagonal(L_batch, dim1=-2, dim2=-1)), dim=1
            )
            out[current_rows] = (
                marginal_constant_t
                - 0.5 * log_n_t
                - 0.5 * logdet_K_batch
                - 0.5 * nu_t * torch.log(residual_batch / 2.0)
            )

    return out

def log_marginal_likelihood_adaptive_torch(G):
    number_models = int(G.shape[0])
    out = torch.empty(number_models, dtype=dtype, device=device)
    active_counts = G.sum(dim=1).to(torch.long)
    active_route_mask = (active_counts + 1) <= int(ADAPTIVE_ACTIVE_MAX_Q)

    if bool(active_route_mask.any().item()):
        idx = torch.nonzero(active_route_mask, as_tuple=False).flatten()
        out[idx] = log_marginal_likelihood_active_torch(G.index_select(0, idx))

    if bool((~active_route_mask).any().item()):
        idx = torch.nonzero(~active_route_mask, as_tuple=False).flatten()
        out[idx] = log_marginal_likelihood_dual_internal_torch(G.index_select(0, idx))

    return out

def log_joint_integrated_torch(G):
    return log_marginal_likelihood_adaptive_torch(G) + log_model_prior_torch(G)


TRAINING_CODE_SHA256 = 'e582664d6c664da509a4547c4fe8fa79a3d635842fa1bdbcec5c04959ba709f2'

def clean_float(x):
    return str(float(x)).replace('.', 'p').replace('-', 'm')

def safe_method_name(method):
    return method.replace('*', 'star').replace('-', '_').replace(' ', '_').lower()

def method_output_dir():
    folder = output_root / f"output_{safe_method_name(METHOD_NAME)}_bodyfat_cython"
    folder.mkdir(parents=True, exist_ok=True)
    return folder

def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False, float_format='%.17g')
    os.replace(temporary, path)


def settings():
    for lr in LR_GRID:
        yield lr, 'annealed', 1, 300, f'gm_annealed_lr_{clean_float(lr)}_{split_label}'


def fit_config(method_seed, train_hash, learning_rate, alpha, S, N):
    return {
        'method': METHOD_NAME, 'method_seed': int(method_seed),
        'tau2': float(tau2), 'a_pi': float(a_pi), 'b_pi': float(b_pi),
        'learning_rate': float(learning_rate), 'alpha_setting': alpha, 'S': int(S), 'N': int(N),
        'LR_GRID': LR_GRID, 'ALPHA_GRID': globals().get('ALPHA_GRID'),
        'PATIENCE_MAX': PATIENCE_MAX, 'MAX_ITER': MAX_ITER,
        'BOUND_TOL': BOUND_TOL, 'MAX_GRAD_NORM': MAX_GRAD_NORM,
        'ADAPTIVE_ACTIVE_MAX_Q': ADAPTIVE_ACTIVE_MAX_Q,
        'ACTIVE_CHOL_BATCH_SIZE': ACTIVE_CHOL_BATCH_SIZE,
        'DUAL_FALLBACK_BATCH_SIZE': DUAL_FALLBACK_BATCH_SIZE,
        'jitter': jitter, 'expected_model_size': PRIOR_EXPECTED_MODEL_SIZE,
        'annealing': ({'start': globals().get('ALPHA_START'), 'minimum': globals().get('ALPHA_MIN'),
                       'step': globals().get('ALPHA_STEP'), 'ess_threshold': globals().get('ESS_LOW')}
                      if 'ALPHA_START' in globals() else None),
        'train_file_sha256': train_hash, 'training_code_sha256': TRAINING_CODE_SHA256,
    }

def vimco_gm_grad_eta(eta, S_copies, N_importance, alpha):
    one_minus_alpha = max(1e-8, 1.0 - float(alpha))
    gamma = gamma_from_eta(eta)
    total_samples = S_copies * N_importance
    G_flat = sample_G(gamma, total_samples)
    logp_flat = log_joint_integrated_torch(G_flat)
    logq_flat = log_q_eta(G_flat, eta)
    logw_flat = logp_flat - logq_flat
    if not bool(torch.isfinite(logw_flat).all().item()):
        raise FloatingPointError("Non-finite VIMCO-GM log importance weight detected.")
    G = G_flat.reshape(S_copies, N_importance, p)
    logw = logw_flat.reshape(S_copies, N_importance)
    logu = one_minus_alpha * logw
    max_logu = torch.max(logu, dim=1, keepdim=True).values
    shifted_logu = logu - max_logu
    u = torch.exp(shifted_logu)
    sum_u = u.sum(dim=1)
    wbar = u / sum_u[:, None]
    bound_per_copy = (max_logu.squeeze(1) + torch.log(sum_u) - np.log(N_importance)) / one_minus_alpha
    train_bound = torch.mean(bound_per_copy)
    ess_per_copy = 1.0 / torch.sum(wbar.square(), dim=1)
    total_ess = torch.sum(ess_per_copy)
    score = G - gamma[None, None, :]
    term1 = -torch.sum(wbar[:, :, None] * score, dim=(0, 1)) / S_copies
    sum_shifted_logu = torch.sum(shifted_logu, dim=1, keepdim=True)
    gm_baseline = torch.exp((sum_shifted_logu - shifted_logu) / (N_importance - 1))
    inside = 1.0 - wbar + gm_baseline / sum_u[:, None]
    inside = torch.clamp(inside, min=1e-12)
    term2 = -torch.sum(score * torch.log(inside)[:, :, None], dim=(0, 1)) / (S_copies * one_minus_alpha)
    grad = term1 + term2
    if not bool(torch.isfinite(grad).all().item()):
        raise FloatingPointError("Non-finite VIMCO-GM gradient detected.")
    return grad, float(train_bound.item()), float(total_ess.item())

def run_vimco_gm(lr, S_copies, N_importance, method_seed):
    torch.manual_seed(int(method_seed))
    np.random.seed(int(method_seed))

    eta = torch.nn.Parameter(make_initial_eta())
    optimizer = torch.optim.Adam([eta], lr=lr, betas=(0.9, 0.999))
    current_alpha = float(ALPHA_START)
    best_training_bound = -float('inf')
    best_eta = None
    patience = 0
    best_iter = 0
    bound_history = []
    alpha_history = []
    ess_history = []

    for it in range(1, MAX_ITER + 1):
        optimizer.zero_grad(set_to_none=True)
        alpha_used = float(current_alpha)

        with torch.no_grad():
            grad, training_bound, ess_value = vimco_gm_grad_eta(
                eta, S_copies, N_importance, alpha_used
            )

        bound_history.append(float(training_bound))
        alpha_history.append(float(alpha_used))
        ess_history.append(float(ess_value))

        if training_bound > best_training_bound + BOUND_TOL:
            best_training_bound = training_bound
            best_eta = eta.detach().clone()
            best_iter = it
            patience = 0
        else:
            if training_bound > best_training_bound:
                best_training_bound = training_bound
                best_eta = eta.detach().clone()
                best_iter = it
            patience += 1

        eta.grad = (-grad).clone()
        torch.nn.utils.clip_grad_norm_([eta], max_norm=MAX_GRAD_NORM)
        optimizer.step()

        if ess_value >= ESS_LOW and it < MAX_ITER:
            alpha_candidate = max(ALPHA_MIN, alpha_used - ALPHA_STEP)
            if alpha_candidate < alpha_used - 1e-12:
                current_alpha = float(alpha_candidate)
                best_training_bound = -float('inf')
                best_eta = None
                best_iter = 0
                patience = 0
            else:
                current_alpha = alpha_used
        else:
            current_alpha = alpha_used

        if patience >= PATIENCE_MAX:
            break


    if best_eta is not None:
        with torch.no_grad():
            eta.copy_(best_eta)

    return {
        'pip': gamma_from_eta(eta).detach().cpu().numpy(),
        'lambda': eta.detach().cpu().numpy(),
        'iterations': int(it),
        'final_alpha': float(current_alpha),
        'best_iter': int(best_iter),
        'saved_alpha': float(alpha_history[best_iter - 1]) if best_iter else np.nan,
        'best_training_bound': float(best_training_bound),
        'bound_history': np.asarray(bound_history, dtype=np.float64),
        'alpha_history': np.asarray(alpha_history, dtype=np.float64),
        'ess_history': np.asarray(ess_history, dtype=np.float64),
        'stop': 'patience' if patience >= PATIENCE_MAX else 'max_iter',
    }


def train_and_save_one_method_seed(method_seed):
    train_path = load_dataset()
    train_hash = hashlib.sha256(Path(train_path).read_bytes()).hexdigest()
    output_dir = method_output_dir()
    stem = f'{safe_method_name(METHOD_NAME)}_{split_label}_methodseed_{method_seed}.csv'
    pip_file = output_dir / f'pips_lambda_{stem}'
    summary_file = output_dir / f'training_summary_{stem}'
    pips = {'variable_name': feature_cols}
    rows = []
    print(f'\n{METHOD_NAME} | method_seed={method_seed} | tau2={tau2:g} | '
          f'S,N={S_COPIES_THIS_FILE},{N_IMPORTANCE_THIS_FILE}', flush=True)
    for lr, alpha, S, N, label in settings():
        config = fit_config(method_seed, train_hash, lr, alpha, S, N)
        signature = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
        print(f'  train {label}', flush=True)
        started = time.perf_counter()
        result = run_vimco_gm(lr, S, N, method_seed)
        runtime = time.perf_counter() - started
        pip = np.asarray(result['pip'], dtype=np.float64)
        eta = np.asarray(result['lambda'], dtype=np.float64)
        if (pip.shape != (p,) or eta.shape != (p,) or not np.isfinite(pip).all()
                or not np.isfinite(eta).all()
                or not np.allclose(pip, 1.0 / (1.0 + np.exp(-eta)), atol=1e-12, rtol=0)):
            raise ValueError(f'Invalid PIPs/logits: method seed {method_seed}, {label}')
        pip_col, lambda_col = f'pip_{label}', f'lambda_{label}'
        pips[pip_col], pips[lambda_col] = pip, eta
        rows.append({
            'method': METHOD_NAME, 'split_label': split_label, 'setting': label,
            'seed': int(method_seed), 'method_seed': int(method_seed),
            'learning_rate': float(lr), 'alpha_setting': alpha,
            'S': int(S), 'N': int(N), 'tau2': float(tau2),
            'a_pi': float(a_pi), 'b_pi': float(b_pi), 'm0': PRIOR_EXPECTED_MODEL_SIZE,
            'runtime_seconds': float(runtime), 'iterations': int(result['iterations']),
            'final_alpha': result['final_alpha'], 'saved_alpha': result.get('saved_alpha', np.nan),
            'best_iter': result.get('best_iter', np.nan),
            'training_bound_saved': result.get('best_training_bound', np.nan),
            'stop': result.get('stop', 'patience' if result['iterations'] < MAX_ITER else 'max_iter'),
            'pip_file': str(pip_file), 'pip_column': pip_col, 'lambda_column': lambda_col,
            'config_sha256': signature, 'train_file_sha256': train_hash,
            'training_code_sha256': TRAINING_CODE_SHA256,
        })
    atomic_csv(pd.DataFrame(pips), pip_file)
    atomic_csv(pd.DataFrame(rows), summary_file)
    return pip_file


for method_seed in METHOD_SEEDS_LIST:
    train_and_save_one_method_seed(method_seed)
print('Done. PIPs, logits and training metadata:', output_root)
